In [2]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder.appName("Data Cleaning Curated 2")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config('spark.driver.memory', '8g')
    .config('spark.executor.memory', '4g')
    .getOrCreate()
)
# Silence any WARN-level logs from this point forth.
spark.sparkContext.setLogLevel("ERROR")

Lets read in our cleaned rental data that has already been merged with `crime rate`.

In [3]:
path = "../../rea_data/analysis/vic_rentals_merged_1.parquet"

df = spark.read.parquet(path)

df.printSchema()

root
 |-- LGA: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- listing_id: string (nullable = true)
 |-- suburb: string (nullable = true)
 |-- weekly_rent: double (nullable = true)
 |-- date_listed: date (nullable = true)
 |-- bedrooms: double (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- carspaces: double (nullable = true)
 |-- property_type: string (nullable = true)
 |-- feat_built_in_wardrobes: integer (nullable = true)
 |-- feat_heating: integer (nullable = true)
 |-- feat_dishwasher: integer (nullable = true)
 |-- feat_secure_parking: integer (nullable = true)
 |-- feat_internal_laundry: integer (nullable = true)
 |-- feat_air_conditioning: integer (nullable = true)
 |-- feat_balcony___deck: integer (nullable = true)
 |-- feat_gas: integer (nullable = true)
 |-- feat_floorboards: integer (nullable = true)
 |-- feat_bath: integer (nullable = true)
 |-- feat_intercom: integer (

<br>
<br>
<br>

### Merging With Population Data

Population data is granularised by SA2 regions, so in order to merge with our curated rental data, we must use the `lat` and `lon` information to calculate which SA2 a given listing lies within. Then merging on SA2 becomes very simple.

##### How we did it:
In the following cell we used the ABS SA2 shapefile which gives each SA2 regions' boundary as a polygon. With geopandas, we turned each property's latitude and longitude into a point and converted it to the shapefile's coordinate system. A spatial join (sjoin with predicate="within") then found which SA2 polygon each point falls inside. 

In [4]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

SA2_FILE = "../../external_data/SA2_2021_AUST_SHP_GDA2020/SA2_2021_AUST_GDA2020.shp"
POP_FILE = "../../external_data/victoria_sa2_population_2012_2025_clean.csv"

Now, reshape the population data from one column per year to one row per SA2 per year, with the columns `SA2_code`, `pop_year` and `population`. Also, change the year to an integer and the SA2 code to text, so both columns can be joined against the listings.

In [5]:
# ---------------------------------------------------------------
# 1. Population: wide -> long (one row per SA2 per year)
# ---------------------------------------------------------------
pop = pd.read_csv(POP_FILE)

# Year columns are the ones named with digits only ("2012" ... "2025")
year_cols = [col for col in pop.columns if col.isdigit()]

# Stack the year columns into rows: SA2_code | pop_year | population
pop = pop.melt(
    id_vars=["SA2 code"],
    value_vars=year_cols,
    var_name="pop_year",
    value_name="population",
)

# Year came from a column name (text), so convert to int for joining
pop["pop_year"] = pop["pop_year"].astype(int)

# Remove the space in the column name
pop = pop.rename(columns={"SA2 code": "SA2_code"})

# Match the shapefile, which stores SA2 codes as text
pop["SA2_code"] = pop["SA2_code"].astype(str)

Now, lets turn each listing's latitude and longitude into a point and finds which Victorian SA2 boundary it falls inside, using the ABS shapefile. The matching SA2 code is added as a new `SA2_code` column, and the last line counts any listings that couldn't be placed in an SA2.

In [6]:
# ---------------------------------------------------------------
# 2. Find each listing's SA2 from its lat/lon
# ---------------------------------------------------------------
# Load SA2 boundaries, keep Victoria only
sa2 = gpd.read_file(SA2_FILE)
sa2 = sa2[sa2["STE_NAME21"] == "Victoria"]

# Turn each listing's lat/lon into a point, in the shapefile's CRS
df_pd = df.toPandas()
points = gpd.GeoDataFrame(
    df_pd,
    geometry=gpd.points_from_xy(
        df_pd["lon"].astype(float), df_pd["lat"].astype(float)
    ),
    crs="EPSG:4326",
).to_crs(sa2.crs)

# Tag each point with the SA2 polygon it falls inside
points = gpd.sjoin(
    points,
    sa2[["SA2_CODE21", "geometry"]],
    how="left",
    predicate="within",
)

# Drop geometry/join helper columns, back to a plain DataFrame
df_pd = pd.DataFrame(points.drop(columns=["geometry", "index_right"]))
df_pd = df_pd.rename(columns={"SA2_CODE21": "SA2_code"})

# Listings outside every SA2 (e.g. bad coordinates)
print(df_pd["SA2_code"].isna().sum(), "listings with no SA2")

0 listings with no SA2


It creates a new `pop_year` column, which is the year used to look up each listing's population. `.clip(upper=2025)` caps the value at 2025. Any Year above 2025 becomes 2025, and everything else stays the same:

In [7]:
# ---------------------------------------------------------------
# 3. Population year
# ---------------------------------------------------------------
df_pd["pop_year"] = df_pd["Year"].clip(upper=2025)


This adds each listing's population by matching its SA2 code and year against the population table, keeping every listing even where no match is found. It then checks that the row count didn't change and counts how many listings came back without a population.

In [8]:
# ---------------------------------------------------------------
# 4. Join on SA2 + year
# ---------------------------------------------------------------
# Count rows first, so we can check the join doesn't duplicate any
rows_before = len(df_pd)

# Add each listing's population, matched on its SA2 and year.
# how="left" keeps every listing, even ones with no match.
df_pd = df_pd.merge(pop, on=["SA2_code", "pop_year"], how="left")

# Row counts should match; any unmatched listings have no population
print("Rows before:", rows_before, "| after:", len(df_pd))
print(df_pd["population"].isna().sum(), "listings with no population")

Rows before: 12316 | after: 12316
0 listings with no population


In [9]:
# ---------------------------------------------------------------
# 5. Back to Spark
# ---------------------------------------------------------------
df_curated_final_merge_2 = spark.createDataFrame(df_pd)
df_curated_final_merge_2.show(1, vertical=True)

-RECORD 0--------------------------------------------------
 LGA                                | Hobsons Bay          
 Year                               | 2026                 
 lat                                | -37.830982           
 lon                                | 144.87091            
 listing_id                         | 16782629             
 suburb                             | SOUTH KINGSVILLE     
 weekly_rent                        | 460.0                
 date_listed                        | 2025-08-13           
 bedrooms                           | 2.0                  
 bathrooms                          | 1.0                  
 carspaces                          | 2.0                  
 property_type                      | Apartment / Unit ... 
 feat_built_in_wardrobes            | 1                    
 feat_heating                       | 1                    
 feat_dishwasher                    | 0                    
 feat_secure_parking                | 1 

Now, lets drop the `SA2_code` and `pop_year` columns because they are no longer needed.

In [10]:
df_curated_final_merge_2 = df_curated_final_merge_2.drop("SA2_code", "pop_year")
df_curated_final_merge_2.printSchema()

root
 |-- LGA: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- listing_id: string (nullable = true)
 |-- suburb: string (nullable = true)
 |-- weekly_rent: double (nullable = true)
 |-- date_listed: date (nullable = true)
 |-- bedrooms: double (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- carspaces: double (nullable = true)
 |-- property_type: string (nullable = true)
 |-- feat_built_in_wardrobes: integer (nullable = true)
 |-- feat_heating: integer (nullable = true)
 |-- feat_dishwasher: integer (nullable = true)
 |-- feat_secure_parking: integer (nullable = true)
 |-- feat_internal_laundry: integer (nullable = true)
 |-- feat_air_conditioning: integer (nullable = true)
 |-- feat_balcony___deck: integer (nullable = true)
 |-- feat_gas: integer (nullable = true)
 |-- feat_floorboards: integer (nullable = true)
 |-- feat_bath: integer (nullable = true)
 |-- feat_intercom: integer (

Lets just do a few final sanity checks to make sure the merge has been done correctly and the numbers make sense.

In [11]:
df = df_curated_final_merge_2

# Missing values
print(df.filter(F.col("population").isNull()).count(), "missing")

# Range
df.select("population").summary("min", "50%", "max").show()

# Listings in SA2s with zero population
print(df.filter(F.col("population") == 0).count(), "with population 0")

# Type
df.select("population").printSchema()

0 missing
+-------+----------+
|summary|population|
+-------+----------+
|    min|        78|
|    50%|     16927|
|    max|     39389|
+-------+----------+

0 with population 0
root
 |-- population: long (nullable = true)



<br>
<br>
<br>

### Saving To Disc

Now, lets save this second merged rental dataset to disc.

In [12]:
OUTPUT_PATH = "../../rea_data/analysis/vic_rentals_merged_2.parquet"

df_curated_final_merge_2.write.mode("overwrite").parquet(OUTPUT_PATH)
print("Saved to", OUTPUT_PATH)

Saved to ../../rea_data/analysis/vic_rentals_merged_2.parquet
